**Analyzing Instegram User Reviews Using BERTopic: Topic Extraction, Hierarchical Clustering, and Trend Analysis**

**Name: K.A.Dayani Anjalika**

**Student id: 66011660**

**Subject: Data science**



In [ ]:
!pip install bertopic[visualization] pandas nltk scikit-learn plotly streamlit

In [ ]:
import kagglehub

path = kagglehub.dataset_download("kundanbedmutha/instagram-analytics-dataset")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'instagram-analytics-dataset' dataset.
Path to dataset files: /kaggle/input/instagram-analytics-dataset


In [ ]:
import os

files = os.listdir(path)
print(files)

['Instagram_Analytics.csv']


In [ ]:
import pandas as pd
import os

file_path = os.path.join(path, "Instagram_Analytics.csv")
df = pd.read_csv(file_path)

df.head()

,post_id,account_id,account_type,follower_count,media_type,content_category,traffic_source,has_call_to_action,post_datetime,post_date,...,comments,shares,saves,reach,impressions,engagement_rate,followers_gained,caption_length,hashtags_count,performance_bucket_label
0,IG0000001,7,brand,3551,reel,Technology,Home Feed,1,2024-11-30 06:00:00,2024-11-30,...,5,7,34,4327,6230,0.0385,899,100,7,medium
1,IG0000002,20,creator,31095,image,Fitness,Hashtags,1,2025-08-15 15:00:00,2025-08-15,...,10,21,68,7451,8268,0.0663,805,122,5,viral
2,IG0000003,15,brand,8167,reel,Beauty,Reels Feed,0,2025-09-11 16:00:00,2025-09-11,...,2,1,22,1639,2616,0.0531,758,115,8,high
3,IG0000004,11,creator,9044,carousel,Music,External,0,2025-09-18 03:00:00,2025-09-18,...,0,7,0,2877,3171,0.0309,402,115,7,medium
4,IG0000005,8,creator,15986,reel,Technology,Profile,0,2025-03-21 09:00:00,2025-03-21,...,8,5,21,5350,8503,0.0221,155,112,9,low


In [ ]:
print(df.columns)

Index(['post_id', 'account_id', 'account_type', 'follower_count', 'media_type',
       'content_category', 'traffic_source', 'has_call_to_action',
       'post_datetime', 'post_date', 'post_hour', 'day_of_week', 'likes',
       'comments', 'shares', 'saves', 'reach', 'impressions',
       'engagement_rate', 'followers_gained', 'caption_length',
       'hashtags_count', 'performance_bucket_label'],
      dtype='object')


In [ ]:
!pip install google-play-scraper

In [ ]:
from google_play_scraper import reviews

result, _ = reviews(
    'com.instagram.android',
    lang='en',
    country='us',
    count=2000
)

In [ ]:
!pip install google-play-scraper bertopic pandas nltk plotly

In [ ]:
from google_play_scraper import reviews
import pandas as pd

result, _ = reviews(
    'com.instagram.android',
    lang='en',
    country='us',
    count=2000
)

df = pd.DataFrame(result)
df.head()

,reviewId,userName,userImage,content,score,thumbsUpCount,reviewCreatedVersion,at,replyContent,repliedAt,appVersion
0,1e9174c1-40d5-4a17-817f-592f4a5b1049,Amit Kumar,https://play-lh.googleusercontent.com/a/ACg8oc...,10 year,5,0,None,2026-03-30 06:10:43,None,None,None
1,3c35beaa-1d0d-4507-8007-0a870d469e96,Kamlesh Raikwar,https://play-lh.googleusercontent.com/a-/ALV-U...,ek number plateform hai unke liye jo time pass...,5,0,None,2026-03-30 06:10:42,None,None,None
2,2070313a-7930-4ed1-a7cc-6f19f5e76524,Manish Rao,https://play-lh.googleusercontent.com/a/ACg8oc...,woww,1,0,422.0.0.44.64,2026-03-30 06:09:30,None,None,422.0.0.44.64
3,5c6ef111-bb39-4bfa-ac6a-c43b9f63b3ec,Akasha A k,https://play-lh.googleusercontent.com/a/ACg8oc...,sopar,5,0,422.0.0.44.64,2026-03-30 06:09:06,None,None,422.0.0.44.64
4,8b890e18-a746-4006-82c1-b6ab03fe257b,Sabirbhai,https://play-lh.googleusercontent.com/a-/ALV-U...,Good use full aap,5,0,422.0.0.44.64,2026-03-30 06:07:02,None,None,422.0.0.44.64


In [ ]:

print(df.columns)

Index(['reviewId', 'userName', 'userImage', 'content', 'score',
       'thumbsUpCount', 'reviewCreatedVersion', 'at', 'replyContent',
       'repliedAt', 'appVersion'],
      dtype='object')


In [ ]:
df = df[['content', 'score', 'at']]

df = df.rename(columns={
    'content': 'text',
    'at': 'date'
})

In [ ]:
import re
import nltk
nltk.download('stopwords')
from nltk.corpus import stopwords

stop_words = set(stopwords.words('english'))

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"http\S+", "", text)
    text = re.sub(r"[^a-z\s]", "", text)
    text = " ".join([w for w in text.split() if w not in stop_words])
    return text

df['clean_text'] = df['text'].apply(clean_text)

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [ ]:
df['date'] = pd.to_datetime(df['date'], errors='coerce')
df = df.dropna(subset=['date'])
df = df.dropna(subset=['clean_text'])

df = df.reset_index(drop=True)

In [ ]:

print(len(df))

2000


In [ ]:
from bertopic import BERTopic

docs = df['clean_text'].tolist()

topic_model = BERTopic(
    verbose=True,
    calculate_probabilities=True
)

topics, probs = topic_model.fit_transform(docs)

2026-03-31 06:11:18,356 - BERTopic - Embedding - Transforming documents to embeddings.


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/63 [00:00<?, ?it/s]

2026-03-31 06:11:39,658 - BERTopic - Embedding - Completed ✓
2026-03-31 06:11:39,660 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-03-31 06:11:51,745 - BERTopic - Dimensionality - Completed ✓
2026-03-31 06:11:51,747 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-03-31 06:11:52,051 - BERTopic - Cluster - Completed ✓
2026-03-31 06:11:52,056 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-03-31 06:11:52,105 - BERTopic - Representation - Completed ✓


In [ ]:
from bertopic import BERTopic
from sentence_transformers import SentenceTransformer

# Better embedding model
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

# Tune BERTopic
topic_model = BERTopic(
    embedding_model=embedding_model,
    min_topic_size=20,        # reduce noise
    n_gram_range=(1, 2),
    calculate_probabilities=True,
    verbose=True
)

topics, probs = topic_model.fit_transform(docs)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
2026-03-31 06:12:00,680 - BERTopic - Embedding - Transforming documents to embeddings.


Batches:   0%|          | 0/63 [00:00<?, ?it/s]

2026-03-31 06:12:14,618 - BERTopic - Embedding - Completed ✓
2026-03-31 06:12:14,620 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-03-31 06:12:26,727 - BERTopic - Dimensionality - Completed ✓
2026-03-31 06:12:26,728 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-03-31 06:12:26,927 - BERTopic - Cluster - Completed ✓
2026-03-31 06:12:26,932 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-03-31 06:12:26,991 - BERTopic - Representation - Completed ✓


In [ ]:
print(df.columns)

Index(['text', 'score', 'date', 'clean_text'], dtype='object')


In [ ]:
topic_model = BERTopic(min_topic_size=10)

In [ ]:
from bertopic import BERTopic
from sentence_transformers import SentenceTransformer

# Use cleaned text
docs = df["clean_text"].astype(str).tolist()

# Create model
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")
topic_model = BERTopic(embedding_model=embedding_model)

# FIT model (important)
topics, probs = topic_model.fit_transform(docs)

# Now works
topic_model.get_topic_info().head(10)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


,Topic,Count,Name,Representation,Representative_Docs
0,-1,282,-1_dont_time_app_like,"[dont, time, app, like, content, instagram, ne...",[disabling see new content contacts pushing ol...
1,0,206,0_hai_bahut_per_ka,"[hai, bahut, per, ka, ho, xx, raha, aur, de, se]",[abaki bar jo update aaya hai vah sahi nahin h...
2,1,128,1_question_haha_amezing_hi,"[question, haha, amezing, hi, much, , , , , ]","[haha, amezing, hi question]"
3,2,112,2_fun_happy_helpful_working,"[fun, happy, helpful, working, useful, use, ad...","[useful fun, nice fun use, best use fun]"
4,3,111,3_good_shes_soo_ever,"[good, shes, soo, ever, , , , , , ]","[good, good, good]"
5,4,95,4_good_yer_kids_goodd,"[good, yer, kids, goodd, vary, yeah, one, , , ]","[good, good, good]"
6,5,70,5_video_reels_upload_story,"[video, reels, upload, story, post, videos, re...",[edit video edits reels link inside video keep...
7,6,69,6_nice_niceee__,"[nice, niceee, , , , , , , , ]","[nice, nice, nice]"
8,7,52,7_nice_app_baddies_appo,"[nice, app, baddies, appo, apppppp, forward, a...","[nice app, nice app, app nice]"
9,8,47,8_nice_woowwww_thats_one,"[nice, woowwww, thats, one, , , , , , ]","[nice, nice, nice]"


In [ ]:
topic_model = BERTopic(
    embedding_model=embedding_model,
    min_topic_size=10,   # smaller = fewer outliers
)

In [ ]:
topics, probs = topic_model.fit_transform(docs)

In [ ]:
new_topics = topic_model.reduce_outliers(docs, topics)

In [ ]:
topic_model.update_topics(docs, topics=new_topics)

2026-03-31 06:13:35,954 - BERTopic - WARNING: Using a custom list of topic assignments may lead to errors if topic reduction techniques are used afterwards. Make sure that manually assigning topics is the last step in the pipeline.Note that topic embeddings will also be created through weightedc-TF-IDF embeddings instead of centroid embeddings.


In [ ]:
from bertopic import BERTopic
from sentence_transformers import SentenceTransformer

# Data
docs = df["clean_text"].astype(str).tolist()
docs = [doc for doc in docs if doc.strip() != ""]

# Model
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")
topic_model = BERTopic(embedding_model=embedding_model)

# STEP 1: Fit
topics, probs = topic_model.fit_transform(docs)

# STEP 2: Reduce outliers
topics = topic_model.reduce_outliers(docs, topics)

# STEP 3: Update model
topic_model.update_topics(docs, topics=topics)

# STEP 4: View topics
topic_info = topic_model.get_topic_info()
topic_info = topic_info[topic_info.Topic != -1]

print(topic_info.head(10))

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
2026-03-31 06:14:18,078 - BERTopic - WARNING: Using a custom list of topic assignments may lead to errors if topic reduction techniques are used afterwards. Make sure that manually assigning topics is the last step in the pipeline.Note that topic embeddings will also be created through weightedc-TF-IDF embeddings instead of centroid embeddings.


    Topic  Count                              Name  \
1       0    218                0_hai_bahut_per_ho   
2       1    158  1_good_intertanment_shes_contain   
3       2    118          2_fun_happy_mast_working   
4       3     87          3_nice_iruinga_niceee_ah   
5       4     88          4_reels_post_video_story   
6       5     73         5_cant_update_fix_problem   
7       6     56             6_good_one_kids_goodd   
8       7     59        7_best_app_favourite_super   
9       8     57                8_nice_app_op_appo   
10      9     52    9_social_friends_media_sharing   

                                       Representation  \
1    [hai, bahut, per, ho, ka, raha, xx, aur, de, hi]   
2   [good, intertanment, shes, contain, disadvanta...   
3   [fun, happy, mast, working, helpful, useful, e...   
4      [nice, iruinga, niceee, ah, vet, safe, , , , ]   
5   [reels, post, video, story, upload, videos, re...   
6   [cant, update, fix, problem, bug, pls, comment...   
7   [g

In [ ]:

# 1. FIT MODEL (DO NOT FORCE REDUCE
topics, probs = topic_model.fit_transform(docs)

# Reduce outliers
topics = topic_model.reduce_outliers(docs, topics)
topic_model.update_topics(docs, topics=topics)


# 2. GET ALL TOPICS

topic_info = topic_model.get_topic_info()

# Remove outliers
topic_info = topic_info[topic_info.Topic != -1]


# 3. REMOVE SIMILAR TOPICS

def get_unique_topics(topic_model, topic_info, top_n=10):
    selected = []
    used_words = set()

    for _, row in topic_info.sort_values(by="Count", ascending=False).iterrows():
        topic = row["Topic"]
        words = [w for w, _ in topic_model.get_topic(topic)[:5]]

        # Check overlap
        overlap = len(set(words) & used_words)

        if overlap < 2:   # allow very little similarity
            selected.append(row)
            used_words.update(words)

        if len(selected) == top_n:
            break

    return pd.DataFrame(selected)

# 4. GET TOP 10 DISTINCT TOPICS

top10_unique = get_unique_topics(topic_model, topic_info, top_n=10)

print(top10_unique)

2026-03-31 06:04:07,023 - BERTopic - WARNING: Using a custom list of topic assignments may lead to errors if topic reduction techniques are used afterwards. Make sure that manually assigning topics is the last step in the pipeline.Note that topic embeddings will also be created through weightedc-TF-IDF embeddings instead of centroid embeddings.


    Topic  Count                                 Name  \
1       0    143  0_good_things_contain_disadvantages   
2       1    121              1_fun_happy_helpful_use   
3       2    115                    2_hai_bahut_ka_xx   
5       4    103            4_reels_video_post_upload   
6       5     78             5_nice_iruinga_niceee_ah   
4       3     77                3_good_one_goodd_kids   
7       6     74               6_kumar_mr_shadab_gram   
12     11     61        11_fix_comments_remove_update   
10      9     56        9_social_friends_media_people   
11     10     56              10_good_app_appp_really   

                                       Representation  \
1   [good, things, contain, disadvantages, safely,...   
2   [fun, happy, helpful, use, working, full, ever...   
3   [hai, bahut, ka, xx, raha, aur, ho, hi, kar, bhi]   
5   [reels, video, post, upload, story, videos, re...   
6      [nice, iruinga, niceee, ah, vet, safe, , , , ]   
4     [good, one, goodd, kids,

In [ ]:
# ===============================
# CLEAN TOPICS FOR FINAL OUTPUT
# ===============================
topic_info = topic_model.get_topic_info()

# Remove outliers
topic_info = topic_info[topic_info.Topic != -1]

# Remove noisy topics (optional filters)
def is_valid_topic(words):
    words = [w for w, _ in words]

    # remove names / random words
    bad_words = ["kumar", "shadab", "hai", "ka", "bahut"]

    for w in words:
        if w in bad_words:
            return False
    return True

clean_topics = []

for _, row in topic_info.iterrows():
    topic = row["Topic"]
    words = topic_model.get_topic(topic)

    if words and is_valid_topic(words):
        clean_topics.append(row)

clean_df = pd.DataFrame(clean_topics)

# ===============================
# REMOVE SIMILAR TOPICS
# ===============================
def remove_similar(df, top_n=10):
    selected = []
    used_words = set()

    for _, row in df.sort_values(by="Count", ascending=False).iterrows():
        topic = row["Topic"]
        words = [w for w, _ in topic_model.get_topic(topic)[:5]]

        overlap = len(set(words) & used_words)

        if overlap < 2:
            selected.append(row)
            used_words.update(words)

        if len(selected) == top_n:
            break

    return pd.DataFrame(selected)

top10_clean = remove_similar(clean_df, top_n=10)

print(top10_clean)

    Topic  Count                                 Name  \
1       0    143  0_good_things_contain_disadvantages   
2       1    121              1_fun_happy_helpful_use   
5       4    103            4_reels_video_post_upload   
6       5     78             5_nice_iruinga_niceee_ah   
4       3     77                3_good_one_goodd_kids   
12     11     61        11_fix_comments_remove_update   
10      9     56        9_social_friends_media_people   
11     10     56              10_good_app_appp_really   
9       8     52           8_best_app_world_favourite   
13     12     50         12_super_superb_aman_efforts   

                                       Representation  \
1   [good, things, contain, disadvantages, safely,...   
2   [fun, happy, helpful, use, working, full, ever...   
5   [reels, video, post, upload, story, videos, re...   
6      [nice, iruinga, niceee, ah, vet, safe, , , , ]   
4     [good, one, goodd, kids, yer, yeah, nice, , , ]   
12  [fix, comments, remove, up

In [ ]:
# ===============================
# IMPROVED DISTINCT TOPIC FILTER
# ===============================
def get_strict_unique_topics(topic_model, topic_info, top_n=10):
    selected = []
    used_words = set()
    used_positive = False   # to allow ONLY ONE "positive" topic

    positive_words = {"good", "nice", "best", "awesome", "great", "love"}

    for _, row in topic_info.sort_values(by="Count", ascending=False).iterrows():
        topic = row["Topic"]
        words = [w for w, _ in topic_model.get_topic(topic)[:5]]

        # Check if topic is mostly positive words
        pos_overlap = len(set(words) & positive_words)

        # 🚫 Skip if it's another generic positive topic
        if pos_overlap >= 2:
            if used_positive:
                continue
            else:
                used_positive = True

        # 🚫 Skip if too similar to already selected topics
        overlap = len(set(words) & used_words)
        if overlap >= 2:
            continue

        selected.append(row)
        used_words.update(words)

        if len(selected) == top_n:
            break

    return pd.DataFrame(selected)

# Apply
top10_clean = get_strict_unique_topics(topic_model, topic_info, top_n=10)

print(top10_clean)

    Topic  Count                                 Name  \
1       0    143  0_good_things_contain_disadvantages   
2       1    121              1_fun_happy_helpful_use   
3       2    115                    2_hai_bahut_ka_xx   
5       4    103            4_reels_video_post_upload   
6       5     78             5_nice_iruinga_niceee_ah   
4       3     77                3_good_one_goodd_kids   
7       6     74               6_kumar_mr_shadab_gram   
12     11     61        11_fix_comments_remove_update   
10      9     56        9_social_friends_media_people   
11     10     56              10_good_app_appp_really   

                                       Representation  \
1   [good, things, contain, disadvantages, safely,...   
2   [fun, happy, helpful, use, working, full, ever...   
3   [hai, bahut, ka, xx, raha, aur, ho, hi, kar, bhi]   
5   [reels, video, post, upload, story, videos, re...   
6      [nice, iruinga, niceee, ah, vet, safe, , , , ]   
4     [good, one, goodd, kids,

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
import pandas as pd

def get_distinct_topics(topic_model, topic_info, top_n=10, similarity_threshold=0.7):
    selected_rows = []
    selected_embeddings = []

    for _, row in topic_info.sort_values(by="Count", ascending=False).iterrows():
        topic_id = row["Topic"]

        # Get topic embedding
        topic_embedding = topic_model.topic_embeddings_[topic_id]

        # Compare with already selected topics
        is_duplicate = False
        for emb in selected_embeddings:
            sim = cosine_similarity([topic_embedding], [emb])[0][0]
            if sim > similarity_threshold:
                is_duplicate = True
                break

        if not is_duplicate:
            selected_rows.append(row)
            selected_embeddings.append(topic_embedding)

        if len(selected_rows) == top_n:
            break

    return pd.DataFrame(selected_rows)


top10_clean = get_distinct_topics(topic_model, topic_info, top_n=10, similarity_threshold=0.7)

print(top10_clean)

    Topic  Count                                 Name  \
1       0    218                   0_hai_bahut_per_ho   
2       1    158     1_good_intertanment_shes_contain   
3       2    118             2_fun_happy_mast_working   
5       4     88             4_reels_post_video_story   
4       3     87             3_nice_iruinga_niceee_ah   
6       5     73            5_cant_update_fix_problem   
10      9     52       9_social_friends_media_sharing   
14     13     51   13_best_ever_forever_entertainment   
15     14     39  14_excellent_beautiful_nice_perfect   
24     23     33        23_password_login_logging_log   

                                       Representation  \
1    [hai, bahut, per, ho, ka, raha, xx, aur, de, hi]   
2   [good, intertanment, shes, contain, disadvanta...   
3   [fun, happy, mast, working, helpful, useful, e...   
5   [reels, post, video, story, upload, videos, re...   
4      [nice, iruinga, niceee, ah, vet, safe, , , , ]   
6   [cant, update, fix, proble

In [ ]:
pip install streamlit

In [ ]:
top10_clean

,Topic,Count,Name,Representation,Representative_Docs
1,0,218,0_hai_bahut_per_ho,"[hai, bahut, per, ho, ka, raha, xx, aur, de, hi]",[abaki bar jo update aaya hai vah sahi nahin h...
2,1,158,1_good_intertanment_shes_contain,"[good, intertanment, shes, contain, disadvanta...","[good, good, good]"
3,2,118,2_fun_happy_mast_working,"[fun, happy, mast, working, helpful, useful, e...","[fun, nice fun use, best use fun]"
5,4,88,4_reels_post_video_story,"[reels, post, video, story, upload, videos, re...",[sooo anooying tried upload pic story add musi...
4,3,87,3_nice_iruinga_niceee_ah,"[nice, iruinga, niceee, ah, vet, safe, , , , ]","[nice, nice, nice]"
6,5,73,5_cant_update_fix_problem,"[cant, update, fix, problem, bug, pls, comment...",[good samsung cant use location sticker lowerc...
10,9,52,9_social_friends_media_sharing,"[social, friends, media, sharing, people, new,...",[amazing app connecting people expressing hone...
14,13,51,13_best_ever_forever_entertainment,"[best, ever, forever, entertainment, better, a...","[best, best, best]"
15,14,39,14_excellent_beautiful_nice_perfect,"[excellent, beautiful, nice, perfect, work, go...","[good excellent, good excellent, excellent good]"
24,23,33,23_password_login_logging_log,"[password, login, logging, log, issue, incorre...",[hello issue logging instagram account tried e...


In [ ]:
top10_clean.to_csv("top10_topics.csv", index=False)

In [ ]:
import streamlit as st
import pandas as pd

# Page config
st.set_page_config(page_title="Topic Modeling Dashboard", layout="wide")

# White background styling
st.markdown("""
    <style>
        body {
            background-color: white;
        }
        .stApp {
            background-color: white;
        }
        .topic-box {
            padding: 15px;
            border-radius: 10px;
            background-color: #f5f5f5;
            margin-bottom: 10px;
        }
    </style>
""", unsafe_allow_html=True)

# Title
st.title("📊 Top 10 BERTopic Topics")

# Load data
df = pd.read_csv("top10_topics.csv")

# Sort properly
df = df.sort_values(by="Count", ascending=False).reset_index(drop=True)

# Display topics
for i, row in df.iterrows():
    st.markdown(f"""
        <div class="topic-box">
            <h4>Topic {i+1}</h4>
            <b>Keywords:</b> {row['Representation']} <br>
            <b>Count:</b> {row['Count']}
        </div>
    """, unsafe_allow_html=True)

2026-03-31 06:17:17.618 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-31 06:17:17.619 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-31 06:17:17.900 
  command:

    streamlit run /usr/local/lib/python3.12/dist-packages/colab_kernel_launcher.py [ARGUMENTS]
2026-03-31 06:17:17.901 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-31 06:17:17.902 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-31 06:17:17.903 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-31 06:17:17.904 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when runn

In [ ]:
!pip install streamlit pyngrok

In [ ]:
!pip install streamlit pyngrok --quiet

In [ ]:
import streamlit as st
import pandas as pd
import ast

# =========================
# PAGE CONFIG
# =========================
st.set_page_config(
    page_title="BERTopic Dashboard",
    layout="wide"
)

# =========================
# CUSTOM WHITE UI
# =========================
st.markdown("""
    <style>
        .stApp {
            background-color: white;
        }
        .title {
            text-align: center;
            font-size: 40px;
            font-weight: bold;
            color: black;
        }
        .topic-box {
            padding: 20px;
            border-radius: 12px;
            background-color: #f7f7f7;
            margin-bottom: 15px;
            box-shadow: 0px 2px 6px rgba(0,0,0,0.1);
        }
    </style>
""", unsafe_allow_html=True)

# =========================
# TITLE
# =========================
st.markdown('<p class="title">📊 Top 10 BERTopic Topics</p>', unsafe_allow_html=True)

# =========================
# LOAD DATA
# =========================
@st.cache_data
def load_data():
    df = pd.read_csv("top10_topics.csv")
    return df

df = load_data()

# =========================
# CLEAN & SORT
# =========================
df = df.sort_values(by="Count", ascending=False).reset_index(drop=True)

# =========================
# DISPLAY TOPICS
# =========================
for i, row in df.iterrows():

    # Convert string list to actual list safely
    try:
        words = ast.literal_eval(row["Representation"])
    except:
        words = row["Representation"]

    # Take top 5 keywords
    if isinstance(words, list):
        keywords = ", ".join(words[:5])
    else:
        keywords = str(words)

    st.markdown(f"""
        <div class="topic-box">
            <h3>🧠 Topic {i+1}</h3>
            <p><b>Keywords:</b> {keywords}</p>
            <p><b>Mentions:</b> {row['Count']}</p>
        </div>
    """, unsafe_allow_html=True)

# =========================
# OPTIONAL BAR CHART
# =========================
st.subheader("📊 Topic Distribution")
st.bar_chart(df.set_index(df.index + 1)["Count"])

2026-03-31 06:24:37.369 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-31 06:24:37.379 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-31 06:24:37.383 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-31 06:24:37.391 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-31 06:24:37.396 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-31 06:24:37.399 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-31 06:24:37.404 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-31 06:24:37.405 No runtime found, using MemoryCacheStorageManager
2026-03-31 06:24:37.409 No runtime found, us

DeltaGenerator()

In [ ]:
import os
os.listdir()

['.config', 'top10_topics.csv', 'sample_data']

In [ ]:
%%writefile app.py
import streamlit as st
import pandas as pd
import ast

st.set_page_config(page_title="BERTopic Dashboard", layout="wide")

st.markdown("""
<style>
.stApp { background-color: white; }
.topic-box {
    padding: 20px;
    border-radius: 12px;
    background-color: #f7f7f7;
    margin-bottom: 15px;
}
</style>
""", unsafe_allow_html=True)

st.title("📊 Top 10 BERTopic Topics")

df = pd.read_csv("top10_topics.csv")
df = df.sort_values(by="Count", ascending=False).reset_index(drop=True)

for i, row in df.iterrows():
    try:
        words = ast.literal_eval(row["Representation"])
        keywords = ", ".join(words[:5])
    except:
        keywords = row["Representation"]

    st.markdown(f"""
    <div class="topic-box">
        <h3>Topic {i+1}</h3>
        <p><b>Keywords:</b> {keywords}</p>
        <p><b>Mentions:</b> {row['Count']}</p>
    </div>
    """, unsafe_allow_html=True)

st.bar_chart(df["Count"])

Writing app.py


In [ ]:
os.listdir()

['.config', 'app.py', 'top10_topics.csv', 'sample_data']

In [ ]:
!streamlit run app.py




  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://34.80.21.178:8501

  Stopping...


In [ ]:
!streamlit run app.py --server.port 8501



2026-03-31 06:31:08.884 Port 8501 is not available


In [ ]:
!pkill -f streamlit

In [ ]:
!streamlit run app.py --server.port 8501




  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://34.80.21.178:8501

  Stopping...


In [ ]:
from google.colab import files

files.download("app.py")
files.download("top10_topics.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

The application is deployed using Streamlit Cloud and accessible via the following link:

https://bertopic-9qblpzjsbh2up88lhqymjq.streamlit.app/

In [ ]:
!pip install bertopic[visualization] sentence-transformers

In [ ]:
from bertopic import BERTopic

In [ ]:
# Hierarchical topics
hierarchical_topics = topic_model.hierarchical_topics(docs)

# Visualize hierarchy
fig = topic_model.visualize_hierarchy(hierarchical_topics=hierarchical_topics)

fig.show()

100%|██████████| 40/40 [00:00<00:00, 126.26it/s]


In [ ]:
print(df.columns)

Index(['Topic', 'Count', 'Name', 'Representation', 'Representative_Docs'], dtype='object')


In [ ]:
from google_play_scraper import reviews
import pandas as pd

result, _ = reviews(
    'com.instagram.android',
    lang='en',
    country='us',
    count=2000
)

df = pd.DataFrame(result)
df.head()

,reviewId,userName,userImage,content,score,thumbsUpCount,reviewCreatedVersion,at,replyContent,repliedAt,appVersion
0,b5e5a6cf-4916-4b93-a471-98f0e6cdc896,Hafizmuhammad Numan mavia,https://play-lh.googleusercontent.com/a-/ALV-U...,I DON'T trust,1,0,None,2026-03-30 08:07:29,None,None,None
1,2726db72-fda3-4d5b-b0cf-98eb0eb620b8,Pramod Chaudhari,https://play-lh.googleusercontent.com/a/ACg8oc...,🔥🔥🔥🔥🔥🔥🔥🔥🔥,5,0,421.0.0.51.66,2026-03-30 08:06:59,None,None,421.0.0.51.66
2,7708c92d-2d4b-40c1-bbe3-b49465908e45,KSM RAMIT,https://play-lh.googleusercontent.com/a-/ALV-U...,❤️🫶,5,0,422.0.0.44.64,2026-03-30 08:06:44,None,None,422.0.0.44.64
3,e33a0205-2299-4fd8-8063-b0a34a447e81,Barsha Khatun,https://play-lh.googleusercontent.com/a/ACg8oc...,jtghdv bg uvxmg uvg yfv ush reel dhan Vishal Neel,5,0,422.0.0.44.64,2026-03-30 08:06:31,None,None,422.0.0.44.64
4,ea4b3835-8298-4b25-b5be-1da342b31332,Krishna CS,https://play-lh.googleusercontent.com/a/ACg8oc...,I can't retrieve my account.,1,0,300.0.0.29.110,2026-03-30 08:06:23,None,None,300.0.0.29.110


In [ ]:
print(df.columns)

Index(['reviewId', 'userName', 'userImage', 'content', 'score',
       'thumbsUpCount', 'reviewCreatedVersion', 'at', 'replyContent',
       'repliedAt', 'appVersion'],
      dtype='object')


In [ ]:
import pandas as pd

df['at'] = pd.to_datetime(df['at'])

In [ ]:
docs = df['content']
topics = topic_model.topics_

In [ ]:
df = df[['content', 'at']].dropna()

In [ ]:
df['at'] = pd.to_datetime(df['at'])

In [ ]:
df = df.reset_index(drop=True)

In [ ]:
docs = df['content'].astype(str)

topics, probs = topic_model.fit_transform(docs)

The topics-over-time visualization reflects how the frequency of specific discussion themes changes over time. While overall review volume may influence the counts, the model highlights which topics gain or lose attention relative to others.

In [ ]:
topics_over_time = topic_model.topics_over_time(
    docs=docs,
    topics=topics,
    timestamps=df['at'],
    nr_bins=10,
    global_tuning=True
)

fig.show()

In [ ]:
hierarchical_topics = topic_model.hierarchical_topics(docs)

fig = topic_model.visualize_hierarchy(hierarchical_topics=hierarchical_topics)
fig.show()

100%|██████████| 48/48 [00:00<00:00, 295.28it/s]


Each box/label = a topic

Lines connecting them = similarity

Closer topics = more related

In [ ]:
fig.write_html("hierarchy.html")

In [1]:
from google.colab import files
uploaded = files.upload()